In [42]:
import pandas as pd

In [43]:
df = pd.read_csv("IMDB Dataset.csv")

In [44]:
df.shape

(50000, 2)

In [45]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [46]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [47]:
df.drop_duplicates(inplace=True)

In [48]:
df.shape

(49582, 2)

# Pre-processing 

### 1. Converting to lowercase

In [49]:
df["review"] = df["review"].str.lower()

### 2. Removing the URLs

In [50]:
import re

#sample_text = "abc is the word, abc" # abc => xyz

#new_text = re.sub("abc", "xyz", sample_text)


In [51]:
def remove_urls(text):
    text = re.sub(r"http\S+", "", text) # (pattern, repl, string) eg - https://www.google.com
    return text

df["review"] = df["review"].apply(remove_urls)

### 3. Removing punctuations

In [52]:
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]", "", text) # A-Z a-z 0-9 |s
    return text

df["review"] = df["review"].apply(remove_punctuations)

### 4. Removing HTML

In [53]:
def remove_html(text):
    text = re.sub(r"<.*?>", "", text)
    return text

df["review"] = df["review"].apply(remove_html)

### 5. Removing the Stopwords

In [54]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\astha\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\astha\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\astha\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [55]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [56]:
#sample_text = "I like coding in python"
#tokens = word_tokenize(sample_text)

In [57]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")

    return text

df["review"] = df["review"].apply(remove_stopwords)

In [58]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


### 6. Stemming

In [59]:
# running -> run
# played -> play
# PorterStemming

from nltk.stem import PorterStemmer

In [60]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

### 7. Encoding 

In [61]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [62]:
y = df["sentiment"]

In [63]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int32

### 8. Vectoriztion

In [64]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,1
1,wder ltle producti br br film techniqu unssum ...,1
2,thought th wder wy spend tme o hot summer week...,1
3,bsclli re fmli lttle boy jke thk re zomb close...,0
4,petter mtte love time mey vulli stunng film wt...,1


In [65]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])

In [66]:
X

<49582x5000 sparse matrix of type '<class 'numpy.float64'>'
	with 4057169 stored elements in Compressed Sparse Row format>

### Dataset & Data Loaders

In [67]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [68]:
X_train.shape

(39665, 5000)

In [69]:
X_test.shape

(9917, 5000)

In [70]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [71]:
X_train

<39665x5000 sparse matrix of type '<class 'numpy.float64'>'
	with 3248334 stored elements in Compressed Sparse Row format>

In [72]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [73]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [ ]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)